# 05 — Análise Comparativa Final (UWaveGestureLibrary)

Carrega todos os `metrics.json` da run mais recente e compara raw / db4 /
learned_wavelet entre os backbones.

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')
sys.path.insert(0, 'config')
from experiment_config import (
    UWAVE_CONFIG, DATA_DIR, RESULTS_DIR, SEED,
    WAVELET_CONFIG, LEARNED_WAVELET_CONFIG, DL_TRAINING_CONFIG,
    ML_MODELS_CONFIG, ML_SEARCH_CONFIG, build_param_dist,
)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.random.seed(SEED)
print('Dataset:', UWAVE_CONFIG['dataset_name'], '| classes:', UWAVE_CONFIG['n_classes'],
      '| seq_len:', UWAVE_CONFIG['sequence_length'], '| canais:', UWAVE_CONFIG['n_features'])

## 1. Carregar resultados da run mais recente

In [ ]:
import json
runs = sorted([p for p in RESULTS_DIR.glob('????-??-??*') if (p/'queue_status.json').exists()], reverse=True)
assert runs, 'Nenhuma run encontrada — rode run_dl_queue.py primeiro.'
run_dir = runs[0]; print('Run:', run_dir.name)
rows = [json.loads(f.read_text()) for f in run_dir.rglob('metrics.json')]
df = pd.json_normalize(rows)
for col in ['test_accuracy','test_f1_macro','test_auc_ovr','val_accuracy']:
    df[col] = pd.to_numeric(df.get(col), errors='coerce')
print(len(df), 'configs avaliados')

## 2. Melhor por Model × Mode

In [ ]:
best = df.groupby(['model_name','mode'])[['test_accuracy','test_f1_macro','test_auc_ovr']].max()
best.sort_values('test_f1_macro', ascending=False).round(4)

## 3. Heatmap f1_macro (Model × Mode)

In [ ]:
import seaborn as sns
pivot = df.groupby(['model_name','mode'])['test_f1_macro'].max().unstack()
order_m = ['raw','db4','learned_wavelet_no_warmup','learned_wavelet']
pivot = pivot.reindex(columns=[c for c in order_m if c in pivot.columns])
plt.figure(figsize=(8,4)); sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn')
plt.title('test_f1_macro'); plt.tight_layout(); plt.show()

## 4. Raw vs Fixed Wavelet vs Learned Wavelet

In [ ]:
def bucket(m):
    if m == 'raw': return 'raw'
    if m == 'db4': return 'fixed_wavelet'
    return 'learned_wavelet'
df['bucket'] = df['mode'].map(bucket)
df.groupby('bucket')[['test_accuracy','test_f1_macro','test_auc_ovr']].agg(['mean','max']).round(4)

## 5. Ranking global (top 15)

In [ ]:
cols = ['model_name','mode','config_idx','test_accuracy','test_f1_macro','test_auc_ovr']
df[cols].sort_values('test_f1_macro', ascending=False).head(15).round(4)